# InstaCart analysis

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Load the dataset
products = pd.read_csv('./data/products.csv')
products.head()

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


In [3]:
import re
import pandas as pd

# Load the dataset (matches your screenshot)
products = pd.read_csv("./data/products.csv")

# -----------------------------
# 1) Keyword / regex dictionaries
# -----------------------------
# Keep these fairly high-precision. You can expand later.
PATTERNS = {
    "diabetes_mgmt": [
        r"\bno\s+added\s+sugar\b",
        r"\bunsweetened\b",
        r"\bsugar[-\s]?free\b",
        r"\bzero\s+sugar\b",
        r"\blow\s+glycemic\b",
        r"\blow\s+gi\b",
        r"\blow\s+carb\b",
        r"\bnet\s+carb(s)?\b",
        r"\bketo\b",
        r"\bhigh\s+fiber\b",
        r"\bwhole\s+grain(s)?\b",
        r"\bwhole\s+wheat\b",
        r"\bdiabetic[-\s]?friendly\b",
    ],
    "hypertension_low_sodium": [
        r"\blow\s+sodium\b",
        r"\bvery\s+low\s+sodium\b",
        r"\breduced\s+sodium\b",
        r"\bno\s+salt\s+added\b",
        r"\bunsalted\b",
        r"\blightly\s+salted\b",
        r"\bsalt[-\s]?free\b",
        r"\bheart\s+healthy\b",
        r"\bdash\b",
    ],
    "gi_gluten_free": [
        r"\bgluten[-\s]?free\b",
        r"\bcertified\s+gluten[-\s]?free\b",
        r"\bwheat[-\s]?free\b",   # note: wheat-free != gluten-free, but still GI-relevant
        r"\bceliac\b",
        r"\blow\s+fodmap\b",
        r"\blactose[-\s]?free\b",
        r"\bdairy[-\s]?free\b",
    ],
}

# Optional: negative / cautionary triggers (helpful for analysis)
# e.g., sugar alcohols can be "diabetes-friendly" but GI-unfriendly for some people
CAUTION = {
    "gi_sensitivity_triggers": [
        r"\binulin\b",
        r"\bchicory\b",
        r"\berythritol\b",
        r"\bsorbitol\b",
        r"\bmannitol\b",
        r"\bxylitol\b",
        r"\bmaltitol\b",
    ]
}

# -----------------------------
# 2) Helper functions
# -----------------------------
def _compile(regex_list):
    return [re.compile(pat, flags=re.IGNORECASE) for pat in regex_list]

COMPILED = {k: _compile(v) for k, v in PATTERNS.items()}
COMPILED_CAUTION = {k: _compile(v) for k, v in CAUTION.items()}

def match_any(text, compiled_regexes):
    if pd.isna(text):
        return False
    t = str(text)
    return any(r.search(t) for r in compiled_regexes)

def matched_keywords(text, compiled_regexes):
    """Return list of regex patterns that matched (for debugging / auditability)."""
    if pd.isna(text):
        return []
    t = str(text)
    hits = []
    for r in compiled_regexes:
        if r.search(t):
            hits.append(r.pattern)
    return hits

# -----------------------------
# 3) Apply to your product names
# -----------------------------
name_col = "product_name"

products["is_diabetes_mgmt"] = products[name_col].apply(lambda x: match_any(x, COMPILED["diabetes_mgmt"]))
products["is_low_sodium_htn"] = products[name_col].apply(lambda x: match_any(x, COMPILED["hypertension_low_sodium"]))
products["is_gi_gluten_free"] = products[name_col].apply(lambda x: match_any(x, COMPILED["gi_gluten_free"]))

# Optional: add "reasons" columns (super useful when you sanity-check labels)
products["diabetes_mgmt_hits"] = products[name_col].apply(lambda x: matched_keywords(x, COMPILED["diabetes_mgmt"]))
products["low_sodium_htn_hits"] = products[name_col].apply(lambda x: matched_keywords(x, COMPILED["hypertension_low_sodium"]))
products["gi_gluten_free_hits"] = products[name_col].apply(lambda x: matched_keywords(x, COMPILED["gi_gluten_free"]))

# Optional: GI sensitivity caution triggers
products["gi_caution_triggers"] = products[name_col].apply(lambda x: matched_keywords(x, COMPILED_CAUTION["gi_sensitivity_triggers"]))
products["has_gi_caution_trigger"] = products["gi_caution_triggers"].apply(lambda xs: len(xs) > 0)

# -----------------------------
# 4) Quick view + save
# -----------------------------
cols = ["product_id", "product_name", "aisle_id", "department_id",
        "is_diabetes_mgmt", "is_low_sodium_htn", "is_gi_gluten_free",
        "has_gi_caution_trigger"]
print(products[cols].head(20))

# Save labeled version
products.to_csv("./data/products_labeled.csv", index=False)

    product_id                                       product_name  aisle_id  \
0            1                         Chocolate Sandwich Cookies        61   
1            2                                   All-Seasons Salt       104   
2            3               Robust Golden Unsweetened Oolong Tea        94   
3            4  Smart Ones Classic Favorites Mini Rigatoni Wit...        38   
4            5                          Green Chile Anytime Sauce         5   
5            6                                       Dry Nose Oil        11   
6            7                     Pure Coconut Water With Orange        98   
7            8                  Cut Russet Potatoes Steam N' Mash       116   
8            9                  Light Strawberry Blueberry Yogurt       120   
9           10     Sparkling Orange Juice & Prickly Pear Beverage       115   
10          11                                  Peach Mango Juice        31   
11          12                         Chocolate Fud

In [5]:
import re
import pandas as pd

# -----------------------------
# 0) Load Instacart tables
# -----------------------------
products = pd.read_csv("./data/products.csv")        # product_id, product_name, aisle_id, department_id
aisles = pd.read_csv("./data/aisles.csv")            # aisle_id, aisle
departments = pd.read_csv("./data/departments.csv")  # department_id, department

# Merge in aisle/department text (big boost for pattern labeling)
df = (
    products.merge(aisles, on="aisle_id", how="left")
            .merge(departments, on="department_id", how="left")
)

# -----------------------------
# 1) Regex dictionaries
# -----------------------------
# Product-name patterns (high precision)
NAME_PATTERNS = {
    "diabetes_mgmt": [
        r"\bno\s+added\s+sugar\b",
        r"\bunsweetened\b",
        r"\bsugar[-\s]?free\b",
        r"\bzero\s+sugar\b",
        r"\blow\s+glycemic\b",
        r"\blow\s+gi\b",
        r"\blow\s+carb\b",
        r"\bnet\s+carb(s)?\b",
        r"\bketo\b",
        r"\bhigh\s+fiber\b",
        r"\bwhole\s+grain(s)?\b",
        r"\bwhole\s+wheat\b",
        r"\bdiabetic[-\s]?friendly\b",
    ],
    "hypertension_low_sodium": [
        r"\blow\s+sodium\b",
        r"\bvery\s+low\s+sodium\b",
        r"\breduced\s+sodium\b",
        r"\bno\s+salt\s+added\b",
        r"\bunsalted\b",
        r"\blightly\s+salted\b",
        r"\bsalt[-\s]?free\b",
        r"\bheart\s+healthy\b",
        r"\bdash\b",
    ],
    "gi_gluten_free": [
        r"\bcertified\s+gluten[-\s]?free\b",
        r"\bgluten[-\s]?free\b",
        r"\bgf\b",
        r"\bwheat[-\s]?free\b",   # GI-relevant but not always gluten-free
        r"\bceliac\b",
        r"\blow\s+fodmap\b",
        r"\blactose[-\s]?free\b",
        r"\bdairy[-\s]?free\b",
    ],
}

# Aisle/department patterns (lower precision than explicit label claims, but very helpful)
# These are intentionally broader; they catch “diet” sections even when product name is generic.
CONTEXT_PATTERNS = {
    "diabetes_mgmt": [
        r"\bdiabetic\b",
        r"\bno\s+sugar\b",
        r"\bsugar[-\s]?free\b",
        r"\blow\s+sugar\b",
        r"\blow\s+carb\b",
        r"\bketo\b",
    ],
    "hypertension_low_sodium": [
        r"\blow\s+sodium\b",
        r"\breduced\s+sodium\b",
        r"\bno\s+salt\b",
        r"\bunsalted\b",
        r"\bheart\s+healthy\b",
        r"\bdash\b",
    ],
    "gi_gluten_free": [
        r"\bgluten[-\s]?free\b",
        r"\bgluten\b",
        r"\bwheat[-\s]?free\b",
        r"\bceliac\b",
        r"\blow\s+fodmap\b",
    ],
}

# Optional: GI caution triggers (often overlap with sugar-free / keto items)
CAUTION_PATTERNS = {
    "gi_sensitivity_triggers": [
        r"\binulin\b",
        r"\bchicory\b",
        r"\berythritol\b",
        r"\bsorbitol\b",
        r"\bmannitol\b",
        r"\bxylitol\b",
        r"\bmaltitol\b",
        r"\bisomalt\b",
    ]
}

# -----------------------------
# 2) Helpers
# -----------------------------
def compile_all(d):
    return {k: [re.compile(p, re.IGNORECASE) for p in v] for k, v in d.items()}

NAME_RE = compile_all(NAME_PATTERNS)
CTX_RE = compile_all(CONTEXT_PATTERNS)
CAUTION_RE = compile_all(CAUTION_PATTERNS)

def norm_text(x):
    if pd.isna(x):
        return ""
    return str(x)

def any_match(text, regs):
    t = norm_text(text)
    return any(r.search(t) for r in regs)

def hits(text, regs):
    t = norm_text(text)
    return [r.pattern for r in regs if r.search(t)]

def context_text(row):
    # Combine aisle + department as “weak signal” context.
    return f"{norm_text(row.get('aisle'))} {norm_text(row.get('department'))}"

# -----------------------------
# 3) Apply labeling logic
# -----------------------------
# We do:
#   - Strong signal: product_name hits (explicit claims)
#   - Weak signal: aisle/department hits
#   - Final label: strong OR weak
#   - Keep provenance columns for auditing.

for key in ["diabetes_mgmt", "hypertension_low_sodium", "gi_gluten_free"]:
    df[f"{key}_name_hits"] = df["product_name"].apply(lambda x: hits(x, NAME_RE[key]))
    df[f"{key}_ctx_hits"] = df.apply(lambda r: hits(context_text(r), CTX_RE[key]), axis=1)

    df[f"is_{key}_by_name"] = df[f"{key}_name_hits"].apply(lambda xs: len(xs) > 0)
    df[f"is_{key}_by_context"] = df[f"{key}_ctx_hits"].apply(lambda xs: len(xs) > 0)

    # Final decision: name OR context
    df[f"is_{key}"] = df[f"is_{key}_by_name"] | df[f"is_{key}_by_context"]

# GI caution triggers (useful to flag “sugar-free” items that may cause GI upset)
df["gi_caution_triggers"] = df["product_name"].apply(lambda x: hits(x, CAUTION_RE["gi_sensitivity_triggers"]))
df["has_gi_caution_trigger"] = df["gi_caution_triggers"].apply(lambda xs: len(xs) > 0)

# -----------------------------
# 4) Friendly column names
# -----------------------------
# Your requested 3 patterns:
df.rename(columns={
    "is_diabetes_mgmt": "is_diabetes_management_pattern",
    "is_hypertension_low_sodium": "is_low_sodium_hypertension_friendly_pattern",
    "is_gi_gluten_free": "is_gi_sensitivity_gluten_free_pattern",
}, inplace=True)

# -----------------------------
# 5) Quick sanity check
# -----------------------------
view_cols = [
    "product_id", "product_name", "aisle_id", "aisle", "department_id", "department",
    "is_diabetes_management_pattern",
    "is_low_sodium_hypertension_friendly_pattern",
    "is_gi_sensitivity_gluten_free_pattern",
    "has_gi_caution_trigger",
]
print(df[view_cols].head(25))

# Show a few examples per label
for col in [
    "is_diabetes_management_pattern",
    "is_low_sodium_hypertension_friendly_pattern",
    "is_gi_sensitivity_gluten_free_pattern",
]:
    print("\n=== Examples for", col, "===")
    ex = df.loc[df[col], ["product_name", "aisle", "department"]].head(10)
    print(ex.to_string(index=False))

# -----------------------------
# 6) Save outputs
# -----------------------------
df.to_csv("./data/products_labeled_instacart.csv", index=False)

# Optional: save a slim file with just labels + ids
slim = df[[
    "product_id",
    "product_name",
    "is_diabetes_management_pattern",
    "is_low_sodium_hypertension_friendly_pattern",
    "is_gi_sensitivity_gluten_free_pattern",
    "has_gi_caution_trigger",
]].copy()
slim.to_csv("./data/products_labels_only.csv", index=False)

    product_id                                       product_name  aisle_id  \
0            1                         Chocolate Sandwich Cookies        61   
1            2                                   All-Seasons Salt       104   
2            3               Robust Golden Unsweetened Oolong Tea        94   
3            4  Smart Ones Classic Favorites Mini Rigatoni Wit...        38   
4            5                          Green Chile Anytime Sauce         5   
5            6                                       Dry Nose Oil        11   
6            7                     Pure Coconut Water With Orange        98   
7            8                  Cut Russet Potatoes Steam N' Mash       116   
8            9                  Light Strawberry Blueberry Yogurt       120   
9           10     Sparkling Orange Juice & Prickly Pear Beverage       115   
10          11                                  Peach Mango Juice        31   
11          12                         Chocolate Fud

In [6]:
import pandas as pd

# Load
orders = pd.read_csv("./data/orders.csv")
prior = pd.read_csv("./data/order_products__prior.csv")
train = pd.read_csv("./data/order_products__train.csv")

products_labeled = pd.read_csv("./data/products_labeled_instacart.csv")

# Combine prior + train purchases
order_products = pd.concat([prior, train], ignore_index=True)

# Merge product labels
order_products = order_products.merge(
    products_labeled[
        [
            "product_id",
            "is_diabetes_management_pattern",
            "is_low_sodium_hypertension_friendly_pattern",
            "is_gi_sensitivity_gluten_free_pattern"
        ]
    ],
    on="product_id",
    how="left"
)

# Merge in user_id
order_products = order_products.merge(
    orders[["order_id", "user_id", "order_number"]],
    on="order_id",
    how="left"
)

In [7]:
# Aggregate to user level
user_stats = (
    order_products
    .groupby("user_id")
    .agg(
        total_products=("product_id", "count"),

        diabetes_count=("is_diabetes_management_pattern", "sum"),
        htn_count=("is_low_sodium_hypertension_friendly_pattern", "sum"),
        gi_count=("is_gi_sensitivity_gluten_free_pattern", "sum"),
    )
    .reset_index()
)

# Compute proportions
user_stats["diabetes_prop"] = user_stats["diabetes_count"] / user_stats["total_products"]
user_stats["htn_prop"] = user_stats["htn_count"] / user_stats["total_products"]
user_stats["gi_prop"] = user_stats["gi_count"] / user_stats["total_products"]

In [8]:
MIN_PURCHASES = 30

user_stats["is_diabetes_pattern_user"] = (
    (user_stats["total_products"] >= MIN_PURCHASES) &
    (user_stats["diabetes_prop"] >= 0.20)
)

user_stats["is_htn_pattern_user"] = (
    (user_stats["total_products"] >= MIN_PURCHASES) &
    (user_stats["htn_prop"] >= 0.20)
)

user_stats["is_gi_pattern_user"] = (
    (user_stats["total_products"] >= MIN_PURCHASES) &
    (user_stats["gi_prop"] >= 0.20)
)

In [11]:
# Order-level flags
order_level = (
    order_products
    .groupby(["user_id", "order_id"])
    .agg(
        diabetes_in_order=("is_diabetes_management_pattern", "max"),
        htn_in_order=("is_low_sodium_hypertension_friendly_pattern", "max"),
        gi_in_order=("is_gi_sensitivity_gluten_free_pattern", "max"),
    )
    .reset_index()
)

order_stats = (
    order_level
    .groupby("user_id")
    .agg(
        total_orders=("order_id", "count"),
        diabetes_orders=("diabetes_in_order", "sum"),
        htn_orders=("htn_in_order", "sum"),
        gi_orders=("gi_in_order", "sum"),
    )
    .reset_index()
)

order_stats["diabetes_order_prop"] = order_stats["diabetes_orders"] / order_stats["total_orders"]
order_stats["htn_order_prop"] = order_stats["htn_orders"] / order_stats["total_orders"]
order_stats["gi_order_prop"] = order_stats["gi_orders"] / order_stats["total_orders"]

user_behavior = user_stats.merge(order_stats, on="user_id")

In [12]:
# -----------------------------
# Consistency thresholds (tune)
# -----------------------------
MIN_PRODUCTS = 30   # minimum total purchased items across (prior+train)
MIN_ORDERS   = 10   # minimum total orders observed

# Diabetes: allow a bit lower product proportion; require sustained order presence
DIAB_PROD_PROP = 0.15
DIAB_ORDER_PROP = 0.30

# HTN / low-sodium: often fewer explicit products per basket -> slightly lower thresholds
HTN_PROD_PROP = 0.10
HTN_ORDER_PROP = 0.25

# GI / gluten-free: can be very consistent for some users; keep similar to diabetes
GI_PROD_PROP = 0.15
GI_ORDER_PROP = 0.30

# -----------------------------
# Consistent user flags
# -----------------------------
user_behavior["is_consistent_diabetes_user"] = (
    (user_behavior["total_products"] >= MIN_PRODUCTS) &
    (user_behavior["total_orders"] >= MIN_ORDERS) &
    (user_behavior["diabetes_prop"] >= DIAB_PROD_PROP) &
    (user_behavior["diabetes_order_prop"] >= DIAB_ORDER_PROP)
)

user_behavior["is_consistent_htn_user"] = (
    (user_behavior["total_products"] >= MIN_PRODUCTS) &
    (user_behavior["total_orders"] >= MIN_ORDERS) &
    (user_behavior["htn_prop"] >= HTN_PROD_PROP) &
    (user_behavior["htn_order_prop"] >= HTN_ORDER_PROP)
)

user_behavior["is_consistent_gi_user"] = (
    (user_behavior["total_products"] >= MIN_PRODUCTS) &
    (user_behavior["total_orders"] >= MIN_ORDERS) &
    (user_behavior["gi_prop"] >= GI_PROD_PROP) &
    (user_behavior["gi_order_prop"] >= GI_ORDER_PROP)
)

In [13]:
user_behavior

,user_id,total_products,diabetes_count,htn_count,gi_count,diabetes_prop,htn_prop,gi_prop,is_diabetes_pattern_user,is_htn_pattern_user,...,total_orders,diabetes_orders,htn_orders,gi_orders,diabetes_order_prop,htn_order_prop,gi_order_prop,is_consistent_diabetes_user,is_consistent_htn_user,is_consistent_gi_user
0,1,70,2,0,0,0.028571,0.000000,0.000000,False,False,...,11,2,0,0,0.181818,0.000000,0.000000,False,False,False
1,2,226,8,6,7,0.035398,0.026549,0.030973,False,False,...,15,4,5,4,0.266667,0.333333,0.266667,False,False,False
2,3,88,15,3,2,0.170455,0.034091,0.022727,False,False,...,12,11,3,2,0.916667,0.250000,0.166667,True,False,False
3,4,18,1,0,0,0.055556,0.000000,0.000000,False,False,...,5,1,0,0,0.200000,0.000000,0.000000,False,False,False
4,5,46,0,2,1,0.000000,0.043478,0.021739,False,False,...,5,0,2,1,0.000000,0.400000,0.200000,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206204,206205,51,2,0,0,0.039216,0.000000,0.000000,False,False,...,4,2,0,0,0.500000,0.000000,0.000000,False,False,False
206205,206206,285,2,0,17,0.007018,0.000000,0.059649,False,False,...,67,2,0,15,0.029851,0.000000,0.223881,False,False,False
206206,206207,223,0,14,0,0.000000,0.062780,0.000000,False,False,...,16,0,9,0,0.000000,0.562500,0.000000,False,False,False
206207,206208,677,24,5,3,0.035451,0.007386,0.004431,False,False,...,49,20,5,3,0.408163,0.102041,0.061224,False,False,False


In [14]:
# count consistent users
consistent_diabetes_users = user_behavior["is_consistent_diabetes_user"].sum()
consistent_htn_users = user_behavior["is_consistent_htn_user"].sum()
consistent_gi_users = user_behavior["is_consistent_gi_user"].sum()
print(f"Consistent diabetes management users: {consistent_diabetes_users}")
print(f"Consistent hypertension-friendly users: {consistent_htn_users}")
print(f"Consistent GI/gluten-free users: {consistent_gi_users}")

Consistent diabetes management users: 977
Consistent hypertension-friendly users: 561
Consistent GI/gluten-free users: 1579


## Temporal analysis

In [29]:
MIN_PRODUCTS = 30   # minimum total purchased items across (prior+train)
MIN_ORDERS   = 10   # minimum total orders observed

# Order-level intensity (proportion of items in that order that are pattern products)

order_pattern = (
    order_products
    .groupby(["user_id", "order_id", "order_number"])
    .agg(
        total_items=("product_id", "count"),
        diabetes_items=("is_diabetes_management_pattern", "sum"),
        htn_items=("is_low_sodium_hypertension_friendly_pattern", "sum"),
        gi_items=("is_gi_sensitivity_gluten_free_pattern", "sum"),
    )
    .reset_index()
)

order_pattern["diabetes_prop"] = order_pattern["diabetes_items"] / order_pattern["total_items"]
order_pattern["htn_prop"] = order_pattern["htn_items"] / order_pattern["total_items"]
order_pattern["gi_prop"] = order_pattern["gi_items"] / order_pattern["total_items"]

# Sort chronologically
order_pattern = order_pattern.sort_values(["user_id", "order_number"])

# Apply MIN_PRODUCTS and MIN_ORDERS filters
user_totals = (
    order_pattern
    .groupby("user_id")
    .agg(
        total_products=("total_items", "sum"),
        total_orders=("order_id", "nunique"),
    )
)
qualified_users = user_totals[
    (user_totals["total_products"] >= MIN_PRODUCTS) &
    (user_totals["total_orders"] >= MIN_ORDERS)
].index

# No... here you filter the order_pattern to only include those qualified users, so that downstream temporal analysis is only on users with enough data.
# but instead of filtering, you should change the compute_temporal_flags function to only compute flags for those qualified users, and return False for others. This way you keep all users in the dataset, but only mark temporal consistency for those with enough data.
order_pattern["qualified_user"] = order_pattern["user_id"].isin(qualified_users)
print(f"Total users: {order_pattern['user_id'].nunique()}")
print(f"Users after MIN_PRODUCTS={MIN_PRODUCTS} / MIN_ORDERS={MIN_ORDERS} filter: {len(qualified_users)}")


Total users: 206209
Users after MIN_PRODUCTS=30 / MIN_ORDERS=10 filter: 105378


In [16]:
import numpy as np

def detect_consistent_state(df, prop_col, T=0.20, N=3, M=3):
    """
    df: one user's orders sorted by order_number
    prop_col: e.g., 'diabetes_prop'
    T: threshold per order
    N: consecutive orders to activate state
    M: required persistence after activation
    """
    signal = (df[prop_col] >= T).astype(int).values
    L = len(signal)

    for start in range(L):
        # Check N consecutive
        if start + N <= L and signal[start:start+N].sum() == N:
            # Activation point
            activation = start

            # After activation, count how long they remain above T
            persistence = 0
            for i in range(activation, L):
                if signal[i] == 1:
                    persistence += 1
                else:
                    break

            # Check persistence
            if persistence >= (N + M):
                return True
            elif activation + persistence == L:
                # End of data — allow if truncated
                return True

    return False

In [30]:
def compute_temporal_flags(order_pattern, prop_col, T, N, M):
    results = []

    for user_id, df in order_pattern.groupby("user_id"):
        df = df.sort_values("order_number")
        # Only run detection for users with enough data; return False for others
        if not df["qualified_user"].iloc[0]:
            flag = False
        else:
            flag = detect_consistent_state(df, prop_col, T=T, N=N, M=M)
        results.append((user_id, flag))

    return pd.DataFrame(results, columns=["user_id", f"is_consistent_{prop_col}_user"])


# Parameters
T = 0.20   # order-level threshold
N = 3      # consecutive orders to activate
M = 3      # must persist 3 more orders

diabetes_temporal = compute_temporal_flags(order_pattern, "diabetes_prop", T, N, M)
htn_temporal = compute_temporal_flags(order_pattern, "htn_prop", T=0.15, N=N, M=M)
gi_temporal = compute_temporal_flags(order_pattern, "gi_prop", T=0.20, N=N, M=M)


In [31]:
user_temporal = (
    diabetes_temporal
    .merge(htn_temporal, on="user_id")
    .merge(gi_temporal, on="user_id")
)

user_temporal.rename(columns={
    "is_consistent_diabetes_prop_user": "is_consistent_diabetes_user",
    "is_consistent_htn_prop_user": "is_consistent_htn_user",
    "is_consistent_gi_prop_user": "is_consistent_gi_user",
}, inplace=True)

#user_temporal.to_csv("./data/user_temporal_consistent_patterns.csv", index=False)

In [32]:
# print the number of consistent users
print(f"Consistent diabetes management users: {user_temporal['is_consistent_diabetes_user'].sum()}")
print(f"Consistent hypertension-friendly users: {user_temporal['is_consistent_htn_user'].sum()}")
print(f"Consistent GI/gluten-free users: {user_temporal['is_consistent_gi_user'].sum()}")

Consistent diabetes management users: 606
Consistent hypertension-friendly users: 206
Consistent GI/gluten-free users: 749


In [34]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Post-activation Kaplan–Meier
# -----------------------------
def first_activation_index(signal: np.ndarray, N: int) -> int:
    """
    Return earliest index i such that signal[i:i+N] are all 1.
    If never activates, return -1.
    """
    L = len(signal)
    if L < N:
        return -1
    win = np.convolve(signal.astype(int), np.ones(N, dtype=int), mode="valid")
    hits = np.where(win == N)[0]
    return int(hits[0]) if len(hits) else -1


def km_durations_post_activation(
    order_pattern: pd.DataFrame,
    in_col: str,
    N: int = 3,
    time_col: str = "order_number",
):
    """
    Build (duration, event) pairs for KM where time starts at *activation*.

    Activation = first time the user achieves N consecutive in-regime orders.

    duration = length of the continuous in-regime run starting at activation
               (in units of `time_col`, typically order_number so it's "orders").

    event = 1 if they ever exit regime after activation (i.e., first 0 occurs)
            0 if right-censored (they stay in regime to end of observed history).

    Only qualified users (qualified_user == True) are included.
    """
    durations = []
    events = []

    df = order_pattern.sort_values(["user_id", "order_number"])

    for user_id, dfu in df.groupby("user_id"):
        # Skip users who don't meet MIN_PRODUCTS / MIN_ORDERS
        if not dfu["qualified_user"].iloc[0]:
            continue

        dfu = dfu.sort_values(time_col).reset_index(drop=True)

        s = dfu[in_col].values.astype(int)
        t = dfu[time_col].values

        act = first_activation_index(s, N)
        if act < 0:
            continue  # never adopted

        # Start KM clock at activation start
        start_t = t[act]

        j = act
        while j < len(s) and s[j] == 1:
            j += 1

        end_t = t[j - 1]
        duration = end_t - start_t + 1  # inclusive count

        # Exit vs censor
        event = 1 if (j < len(s) and s[j] == 0) else 0

        durations.append(float(duration))
        events.append(int(event))

    return np.array(durations, dtype=float), np.array(events, dtype=int)


def kaplan_meier(durations: np.ndarray, events: np.ndarray):
    """
    Simple Kaplan–Meier step estimator.
    Returns arrays (times, survival_probs).
    """
    if len(durations) == 0:
        return np.array([]), np.array([])

    order = np.argsort(durations)
    d = durations[order]
    e = events[order]

    uniq = np.unique(d)
    n_at_risk = len(d)
    S = 1.0

    times = [0.0]
    surv = [1.0]

    for tt in uniq:
        idx = (d == tt)
        d_i = int(e[idx].sum())            # exits at tt
        c_i = int(idx.sum() - d_i)         # censored at tt

        if n_at_risk > 0:
            if d_i > 0:
                S *= (1.0 - d_i / n_at_risk)
            times.append(float(tt))
            surv.append(float(S))

        n_at_risk -= (d_i + c_i)

    return np.array(times), np.array(surv)


def plot_post_activation_km(
    order_pattern: pd.DataFrame,
    regimes: dict,
    time_col: str = "order_number",
    out_png: str = "./data/phase2_km_survival_post_activation_2.png",
):
    """
    regimes example:
      {
        "diabetes": {"prop_col":"diabetes_prop","T":0.20,"N":3},
        "htn": {"prop_col":"htn_prop","T":0.15,"N":3},
        "gi": {"prop_col":"gi_prop","T":0.20,"N":3},
      }
    """
    df = order_pattern.copy()

    # build in-regime columns
    for rname, cfg in regimes.items():
        df[f"in_{rname}"] = (df[cfg["prop_col"]] >= cfg["T"]).astype(int)

    plt.figure()
    for rname, cfg in regimes.items():
        durations, events = km_durations_post_activation(
            df, in_col=f"in_{rname}", N=cfg["N"], time_col=time_col
        )
        t, s = kaplan_meier(durations, events)
        if len(t):
            plt.step(t, s, where="post", label=f"{rname} (n={len(durations)})")

    plt.title("Kaplan–Meier: Persistence After Activation (post-N consecutive)")
    plt.xlabel("Duration after activation (orders)")
    plt.ylabel("Survival prob (still in regime)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()
    print("Saved:", out_png)


# -----------------------------
# Example call (matches your earlier thresholds)
# -----------------------------
REGIMES = {
    "diabetes": {"prop_col": "diabetes_prop", "T": 0.20, "N": 3},
    "htn":      {"prop_col": "htn_prop",      "T": 0.15, "N": 3},
    "gi":       {"prop_col": "gi_prop",       "T": 0.20, "N": 3},
}

# Assuming you already have `order_pattern` built (as in earlier code):
plot_post_activation_km(order_pattern, REGIMES)


Saved: ./data/phase2_km_survival_post_activation_2.png
